# 05c: MLP Analysis: Circuit Mean Ablation

## Overview
Analyzes MLP sub-layer representations under **mean ablation** (circuit mode).
Non-circuit edges have their outputs replaced with dataset-mean activations,
isolating the computational path through the ACDC-discovered circuit.
This is the circuit counterpart to NB05 (base MLP analysis).

## Key Questions
1. How do MLP output norms change under circuit ablation?
2. Does the circuit preserve frequency-band separability in MLP outputs?
3. How does MLP contribution magnitude change under ablation?
4. Are frequency-selective neurons preserved in the circuit?
5. Does MLP activation sparsity change under ablation?

## Hypothesis Domain: R5c (Circuit MLP Contributions)
- **H-R5c.1**: Circuit MLP norm profiles mirror base norms for in-circuit layers
- **H-R5c.2**: Circuit MLP probe accuracy is lower than base but above chance
- **H-R5c.3**: Neuron selectivity is preserved for in-circuit MLP layers
- **H-R5c.4**: Circuit MLP sparsity patterns correlate with base sparsity

## Notebook Structure
1. Setup & Data Loading
2. Circuit MLP Output Geometry (norms)
3. Circuit MLP Separability (linear probing)
4. Circuit MLP Logit Contribution (norm proxy)
5. Circuit Neuron Selectivity
6. Circuit MLP Sparsity
7. Base vs Circuit Comparison
8. Cross-Model Patterns

## Data Sources
- Circuit activations: `outputs/extraction/circuit_activations/` (NPZ with `mlp_out_predpos`, `mlp_pre_predpos`)
- Base MLP analysis: `outputs/mlp/base/analysis/` (05_*.csv)
- Prune scores: circuit discovery outputs for in-circuit masks

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial
from scipy import stats
from sklearn.decomposition import PCA

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    MODEL_D_MLP,
    RANDOM_SEED,
    CV_FOLDS,
    get_domain_dirs,
)
from utils.data_loading import save_analysis, load_domain_csv
from utils.circuit_loading import (
    load_circuit_activations,
    load_base_and_circuit,
    load_prune_scores,
    get_circuit_mask,
)
from utils.geometry import (
    compute_band_centroids,
    compute_separation_ratio,
    compute_within_band_spread,
    compute_centroid_distances,
)
from utils.probing import train_probe
from utils.plotting import setup_plotting, save_figure

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()

CIRCUIT_ANALYSIS, CIRCUIT_VIZ = get_domain_dirs("mlp", "circuit")
COMP_ANALYSIS, COMP_VIZ = get_domain_dirs("mlp", "comparison")
save_analysis_circuit = _partial(save_analysis, analysis_dir=CIRCUIT_ANALYSIS)
save_figure_circuit = _partial(save_figure, viz_dir=CIRCUIT_VIZ)
save_analysis_comp = _partial(save_analysis, analysis_dir=COMP_ANALYSIS)
save_figure_comp = _partial(save_figure, viz_dir=COMP_VIZ)

PROBE_PCA_DIM = 50  # PCA dimensionality reduction before probing (speed optimization)

print(f"Models: {MODELS}")
print(f"Bands:  {BANDS}")
print(f"Draws:  {DRAWS}")
print(f"Circuit analysis dir: {CIRCUIT_ANALYSIS}")
print(f"Comparison analysis dir: {COMP_ANALYSIS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands:  ['low', 'medium', 'high', 'very_high', 'control']
Draws:  ['draw_1', 'draw_2', 'draw_3']
Circuit analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/mlp/circuit/analysis
Comparison analysis dir: LSC_circuit_analysis/03_Phase_Representational/outputs/mlp/comparison/analysis


## 2. Circuit MLP Output Geometry

Compute L2 norms of `mlp_out_predpos` per layer per band under circuit mean ablation.
This reveals how much the MLP sub-layer contributes in magnitude when only
in-circuit edges are active.

In [2]:
rows_norms = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for band in BANDS:
        for draw in DRAWS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            mlp_out = data["mlp_out_predpos"]  # (N, n_layers, d_model)

            for layer in range(n_layers):
                X = mlp_out[:, layer, :]  # (N, d_model)
                norms = np.linalg.norm(X, axis=1)

                rows_norms.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "layer": layer,
                        "mean_norm": float(norms.mean()),
                        "std_norm": float(norms.std()),
                        "median_norm": float(np.median(norms)),
                    }
                )

df_circuit_norms = pd.DataFrame(rows_norms)
save_analysis_circuit(df_circuit_norms, "05c_circuit_mlp_norms.csv")
print(f"Circuit MLP norm records: {len(df_circuit_norms)}")
df_circuit_norms.head()

Circuit MLP norm records: 1230


,model,band,draw,layer,mean_norm,std_norm,median_norm
0,pythia-70m,low,draw_1,0,6.798147,0.567965,6.762117
1,pythia-70m,low,draw_1,1,5.311841,1.467718,4.780629
2,pythia-70m,low,draw_1,2,12.025826,14.179291,5.631016
3,pythia-70m,low,draw_1,3,9.013937,1.506899,8.522376
4,pythia-70m,low,draw_1,4,15.844097,9.879573,11.818924


In [3]:
# Visualization: circuit MLP output norm trajectory per model, colored by band
for model in MODELS:
    df_m = df_circuit_norms[df_circuit_norms["model"] == model]
    if df_m.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        df_b = (
            df_m[df_m["band"] == band]
            .groupby("layer")
            .mean(numeric_only=True)
            .reset_index()
        )
        if df_b.empty:
            continue
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        ax.plot(
            df_b["layer"],
            df_b["mean_norm"],
            color=color,
            label=label,
            marker="o",
            markersize=3,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean L2 Norm (MLP Output)")
    ax.set_title(f"Circuit MLP Output Norm Trajectory \u2014 {model}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_05c_01_circuit_mlp_norms_{model}.png")

## 3. Circuit MLP Separability

Linear probe on circuit-mode `mlp_out_predpos` per layer.
Can the circuit MLP outputs alone still classify frequency band?

In [4]:
rows_probe = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]
    n_pca = min(PROBE_PCA_DIM, d_model)
    print(f"\n{model} ({n_layers} layers, d_model={d_model}, PCA->{n_pca}):")

    for draw in DRAWS:
        # Collect all bands for this model/draw
        band_data = {}
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
                band_data[band] = data["mlp_out_predpos"]  # (N, n_layers, d_model)
            except FileNotFoundError:
                pass

        if len(band_data) < 2:
            continue

        for layer in range(n_layers):
            all_X = []
            all_labels = []
            for band, mlp_out in band_data.items():
                all_X.append(mlp_out[:, layer, :])
                all_labels.extend([band] * mlp_out.shape[0])

            all_X = np.vstack(all_X)
            all_labels = np.array(all_labels)

            # PCA dimensionality reduction for speed
            if all_X.shape[1] > n_pca:
                pca = PCA(n_components=n_pca, random_state=RANDOM_SEED)
                all_X = pca.fit_transform(all_X)

            result = train_probe(all_X, all_labels, n_folds=CV_FOLDS)
            rows_probe.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "accuracy": result["accuracy"],
                    "std": result["std"],
                }
            )

        # Report peak for this draw
        draw_rows = [r for r in rows_probe if r["model"] == model and r["draw"] == draw]
        if draw_rows:
            peak = max(draw_rows, key=lambda r: r["accuracy"])
            print(
                f"  {draw}: peak circuit MLP probe = {peak['accuracy']:.3f} at layer {peak['layer']}"
            )

df_circuit_probe = pd.DataFrame(rows_probe)
save_analysis_circuit(df_circuit_probe, "05c_circuit_mlp_probe.csv")
print(f"\nCircuit MLP probe records: {len(df_circuit_probe)}")


pythia-70m (6 layers, d_model=512, PCA->50):


  draw_1: peak circuit MLP probe = 0.532 at layer 4


  draw_2: peak circuit MLP probe = 0.531 at layer 2


  draw_3: peak circuit MLP probe = 0.525 at layer 5

pythia-160m (12 layers, d_model=768, PCA->50):


  draw_1: peak circuit MLP probe = 0.629 at layer 8


...

  draw_2: peak circuit MLP probe = 0.700 at layer 15


  draw_3: peak circuit MLP probe = 0.670 at layer 14

pythia-1.4b (24 layers, d_model=2048, PCA->50):


  draw_1: peak circuit MLP probe = 0.660 at layer 20


  draw_2: peak circuit MLP probe = 0.733 at layer 23


  draw_3: peak circuit MLP probe = 0.727 at layer 22

Circuit MLP probe records: 246


In [5]:
# Visualization: circuit MLP probe trajectory per model
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    df_m = df_circuit_probe[df_circuit_probe["model"] == model]
    if df_m.empty:
        continue
    df_mean = (
        df_m.groupby("layer").agg({"accuracy": "mean", "std": "mean"}).reset_index()
    )
    color = MODEL_COLORS.get(model, "gray")
    ax.plot(
        df_mean["layer"],
        df_mean["accuracy"],
        color=color,
        label=model,
        marker="o",
        ms=4,
    )
    ax.fill_between(
        df_mean["layer"],
        df_mean["accuracy"] - df_mean["std"],
        df_mean["accuracy"] + df_mean["std"],
        color=color,
        alpha=0.15,
    )

ax.axhline(
    y=1.0 / len(BANDS), color="gray", linestyle="--", alpha=0.5, label="Chance (1/5)"
)
ax.set_xlabel("Layer")
ax.set_ylabel("Probe Accuracy")
ax.set_title("Circuit MLP Probe Trajectory")
ax.legend()
ax.set_ylim(0, 1)
fig.tight_layout()
save_figure_circuit(fig, "viz_05c_02_circuit_mlp_probe_trajectory.png")

## 4. Circuit MLP Logit Contribution

Without direct access to the unembedding matrix W_U in this notebook,
we use the **norm of `mlp_out_predpos`** as a proxy for contribution magnitude.
Larger MLP output norms indicate stronger potential influence on the logit space.

In [6]:
rows_contrib = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for band in BANDS:
        for draw in DRAWS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            mlp_out = data["mlp_out_predpos"]  # (N, n_layers, d_model)

            for layer in range(n_layers):
                X = mlp_out[:, layer, :]  # (N, d_model)
                norms = np.linalg.norm(X, axis=1)

                rows_contrib.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "layer": layer,
                        "mean_contribution": float(norms.mean()),
                        "std_contribution": float(norms.std()),
                        "median_contribution": float(np.median(norms)),
                        "method": "norm_proxy",
                    }
                )

df_circuit_contrib = pd.DataFrame(rows_contrib)
save_analysis_circuit(df_circuit_contrib, "05c_circuit_mlp_contribution.csv")
print(f"Circuit MLP contribution records: {len(df_circuit_contrib)}")
df_circuit_contrib.head()

Circuit MLP contribution records: 1230


,model,band,draw,layer,mean_contribution,std_contribution,median_contribution,method
0,pythia-70m,low,draw_1,0,6.798147,0.567965,6.762117,norm_proxy
1,pythia-70m,low,draw_1,1,5.311841,1.467718,4.780629,norm_proxy
2,pythia-70m,low,draw_1,2,12.025826,14.179291,5.631016,norm_proxy
3,pythia-70m,low,draw_1,3,9.013937,1.506899,8.522376,norm_proxy
4,pythia-70m,low,draw_1,4,15.844097,9.879573,11.818924,norm_proxy


In [7]:
# Visualization: circuit MLP contribution (norm proxy) trajectory per model
for model in MODELS:
    df_m = df_circuit_contrib[df_circuit_contrib["model"] == model]
    if df_m.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        df_b = (
            df_m[df_m["band"] == band]
            .groupby("layer")
            .mean(numeric_only=True)
            .reset_index()
        )
        if df_b.empty:
            continue
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        ax.plot(
            df_b["layer"],
            df_b["mean_contribution"],
            color=color,
            label=label,
            marker="o",
            markersize=3,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("MLP Output Norm (Contribution Proxy)")
    ax.set_title(f"Circuit MLP Contribution Magnitude \u2014 {model}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_05c_03_circuit_mlp_contribution_{model}.png")

## 5. Circuit Neuron Selectivity

From `mlp_pre_predpos` under circuit ablation: compute a selectivity index
for each neuron. The selectivity index is the ratio of the maximum band-mean
activation to the overall mean activation. Neurons with selectivity > 2x
are counted as frequency-selective.

In [8]:
rows_selectivity = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_mlp = MODEL_D_MLP[model]
    print(f"\n{model} (d_mlp={d_mlp}, {n_layers} layers):")

    for draw in ["draw_1"]:  # Single draw for efficiency
        # Collect mlp_pre activations per band
        all_pre_by_band = {}
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
                if "mlp_pre_predpos" in data:
                    all_pre_by_band[band] = data[
                        "mlp_pre_predpos"
                    ]  # (N, n_layers, d_mlp)
            except FileNotFoundError:
                pass

        if len(all_pre_by_band) < 2:
            print("  Skipping: insufficient bands with mlp_pre data")
            continue

        for layer in range(n_layers):
            # Gather per-band neuron activations at this layer
            band_means = {}  # band -> mean abs activation per neuron (d_mlp,)
            all_acts = []
            all_labels = []
            for band in BANDS:
                if band not in all_pre_by_band:
                    continue
                pre = all_pre_by_band[band][:, layer, :]  # (N_band, d_mlp)
                band_means[band] = np.abs(pre).mean(axis=0)  # (d_mlp,)
                all_acts.append(pre)
                all_labels.extend([band] * pre.shape[0])

            if len(band_means) < 2:
                continue

            # Selectivity index per neuron: max band activation / mean activation
            band_mean_arr = np.stack(
                list(band_means.values()), axis=0
            )  # (n_bands, d_mlp)
            max_band_act = band_mean_arr.max(axis=0)  # (d_mlp,)
            overall_mean_act = band_mean_arr.mean(axis=0)  # (d_mlp,)
            # Avoid division by zero
            safe_mean = np.maximum(overall_mean_act, 1e-10)
            selectivity_index = max_band_act / safe_mean  # (d_mlp,)

            # Count frequency-selective neurons (selectivity > 2x)
            n_selective = int(np.sum(selectivity_index > 2.0))
            n_highly_selective = int(np.sum(selectivity_index > 3.0))

            # Also compute ANOVA F-statistic for comparison with base
            all_acts_stacked = np.vstack(all_acts)
            all_labels_arr = np.array(all_labels)
            unique_labels = sorted(set(all_labels_arr))
            label_to_int = {l: i for i, l in enumerate(unique_labels)}
            label_ints = np.array([label_to_int[l] for l in all_labels_arr])

            # Sample F-statistics (subsample neurons for speed)
            n_sample_neurons = min(d_mlp, 500)
            rng = np.random.RandomState(RANDOM_SEED)
            sample_idx = rng.choice(d_mlp, n_sample_neurons, replace=False)
            f_stats_sample = np.zeros(n_sample_neurons)
            for i, nidx in enumerate(sample_idx):
                groups = [
                    all_acts_stacked[label_ints == j, nidx]
                    for j in range(len(unique_labels))
                ]
                valid_groups = [g for g in groups if len(g) >= 2]
                if len(valid_groups) >= 2:
                    try:
                        f_stat, _ = stats.f_oneway(*valid_groups)
                        if np.isfinite(f_stat):
                            f_stats_sample[i] = f_stat
                    except Exception:
                        pass

            rows_selectivity.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "mean_selectivity_index": float(selectivity_index.mean()),
                    "median_selectivity_index": float(np.median(selectivity_index)),
                    "max_selectivity_index": float(selectivity_index.max()),
                    "n_selective_2x": n_selective,
                    "n_highly_selective_3x": n_highly_selective,
                    "frac_selective_2x": n_selective / d_mlp,
                    "frac_selective_3x": n_highly_selective / d_mlp,
                    "mean_f_stat_sample": float(f_stats_sample.mean()),
                    "median_f_stat_sample": float(np.median(f_stats_sample)),
                }
            )

        print(f"  Done: {n_layers} layers")

df_circuit_selectivity = pd.DataFrame(rows_selectivity)
save_analysis_circuit(df_circuit_selectivity, "05c_circuit_neuron_selectivity.csv")
print(f"\nCircuit neuron selectivity records: {len(df_circuit_selectivity)}")
df_circuit_selectivity.head()


pythia-70m (d_mlp=2048, 6 layers):


  Done: 6 layers

pythia-160m (d_mlp=3072, 12 layers):


  Done: 12 layers

pythia-410m (d_mlp=4096, 24 layers):


  Done: 24 layers

pythia-1b (d_mlp=8192, 16 layers):


  Done: 16 layers

pythia-1.4b (d_mlp=8192, 24 layers):


  Done: 24 layers

Circuit neuron selectivity records: 82


,model,draw,layer,mean_selectivity_index,median_selectivity_index,max_selectivity_index,n_selective_2x,n_highly_selective_3x,frac_selective_2x,frac_selective_3x,mean_f_stat_sample,median_f_stat_sample
0,pythia-70m,draw_1,0,1.143970,1.136083,1.475312,0,0,0.0,0.0,16.727187,13.124373
1,pythia-70m,draw_1,1,1.112939,1.097627,1.564256,0,0,0.0,0.0,8.689429,4.479219
2,pythia-70m,draw_1,2,1.115625,1.101484,1.433305,0,0,0.0,0.0,8.617998,5.153786
3,pythia-70m,draw_1,3,1.111362,1.097837,1.635316,0,0,0.0,0.0,9.009312,4.338735
4,pythia-70m,draw_1,4,1.117127,1.101571,1.528581,0,0,0.0,0.0,9.474644,5.141703


In [9]:
# Visualization: fraction of selective neurons per layer, across models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: fraction selective (selectivity > 2x)
ax = axes[0]
for model in MODELS:
    md = df_circuit_selectivity[df_circuit_selectivity["model"] == model].sort_values(
        "layer"
    )
    if md.empty:
        continue
    ax.plot(
        md["layer"],
        md["frac_selective_2x"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
ax.set_xlabel("Layer")
ax.set_ylabel("Fraction Selective (>2x)")
ax.set_title("Circuit: Frequency-Selective Neurons per Layer")
ax.legend(fontsize=8)

# Right: mean selectivity index
ax = axes[1]
for model in MODELS:
    md = df_circuit_selectivity[df_circuit_selectivity["model"] == model].sort_values(
        "layer"
    )
    if md.empty:
        continue
    ax.plot(
        md["layer"],
        md["mean_selectivity_index"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
ax.set_xlabel("Layer")
ax.set_ylabel("Mean Selectivity Index")
ax.set_title("Circuit: Mean Neuron Selectivity per Layer")
ax.legend(fontsize=8)

fig.tight_layout()
save_figure_circuit(fig, "viz_05c_04_circuit_neuron_selectivity.png")

## 6. Circuit MLP Sparsity

Gini coefficient of `mlp_pre` activation magnitudes per band under circuit ablation.
Higher Gini indicates more sparse (concentrated) activations.

In [10]:
def gini_coefficient(x):
    """Compute Gini coefficient of an array of values.

    Higher values indicate more inequality (sparsity).
    Returns value in [0, 1].
    """
    x = np.abs(x)
    if np.sum(x) == 0:
        return 0.0
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * x) / (n * np.sum(x))) - (n + 1) / n)

In [11]:
rows_sparsity = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for band in BANDS:
        for draw in DRAWS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            if "mlp_pre_predpos" not in data:
                continue

            mlp_pre = data["mlp_pre_predpos"]  # (N, n_layers, d_mlp)

            for layer in range(n_layers):
                X = mlp_pre[:, layer, :]  # (N, d_mlp)
                N = X.shape[0]

                # Compute Gini for each example and average
                gini_vals = [gini_coefficient(X[i]) for i in range(N)]

                # Population-level Gini (across mean activation magnitude per neuron)
                mean_abs_act = np.abs(X).mean(axis=0)  # (d_mlp,)
                pop_gini = gini_coefficient(mean_abs_act)

                rows_sparsity.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_gini": float(np.mean(gini_vals)),
                        "std_gini": float(np.std(gini_vals)),
                        "population_gini": pop_gini,
                        "mean_abs_activation": float(np.abs(X).mean()),
                    }
                )

df_circuit_sparsity = pd.DataFrame(rows_sparsity)
save_analysis_circuit(df_circuit_sparsity, "05c_circuit_mlp_sparsity.csv")
print(f"Circuit MLP sparsity records: {len(df_circuit_sparsity)}")
df_circuit_sparsity.head()

Circuit MLP sparsity records: 1230


,model,draw,band,layer,mean_gini,std_gini,population_gini,mean_abs_activation
0,pythia-70m,draw_1,low,0,0.424097,0.005911,0.103976,0.298893
1,pythia-70m,draw_1,low,1,0.418595,0.009795,0.281578,0.867732
2,pythia-70m,draw_1,low,2,0.484518,0.018072,0.343801,0.623602
3,pythia-70m,draw_1,low,3,0.439565,0.005828,0.318125,0.686138
4,pythia-70m,draw_1,low,4,0.413199,0.018391,0.261086,0.735015


In [12]:
# Visualization: circuit Gini coefficient trajectory per model, colored by band
for model in MODELS:
    df_m = df_circuit_sparsity[
        (df_circuit_sparsity["model"] == model)
        & (df_circuit_sparsity["draw"] == "draw_1")
    ]
    if df_m.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    for band in BANDS:
        bd = df_m[df_m["band"] == band].sort_values("layer")
        if bd.empty:
            continue
        ax.plot(
            bd["layer"],
            bd["mean_gini"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean Gini Coefficient")
    ax.set_title(f"Circuit MLP Activation Sparsity \u2014 {model}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_05c_05_circuit_sparsity_{model}.png")

## 7. Base vs Circuit Comparison

Load base MLP analysis results and overlay with circuit results:
- MLP norm trajectory (base vs circuit)
- MLP probe trajectory (base vs circuit)
- Neuron selectivity correlation
- Sparsity comparison

In [13]:
# Load base MLP analysis CSVs
try:
    df_base_norms = load_domain_csv("mlp", "base", "05_mlp_output_norms.csv")
    print(f"Base MLP norms: {len(df_base_norms)} rows")
except FileNotFoundError:
    df_base_norms = pd.DataFrame()
    print("Base MLP norms not found")

try:
    df_base_probe = load_domain_csv("mlp", "base", "05_mlp_probe_trajectory.csv")
    print(f"Base MLP probe: {len(df_base_probe)} rows")
except FileNotFoundError:
    df_base_probe = pd.DataFrame()
    print("Base MLP probe not found")

try:
    df_base_selectivity = load_domain_csv(
        "mlp", "base", "05_neuron_selectivity_summary.csv"
    )
    print(f"Base neuron selectivity: {len(df_base_selectivity)} rows")
except FileNotFoundError:
    df_base_selectivity = pd.DataFrame()
    print("Base neuron selectivity not found")

try:
    df_base_sparsity = load_domain_csv("mlp", "base", "05_mlp_sparsity.csv")
    print(f"Base MLP sparsity: {len(df_base_sparsity)} rows")
except FileNotFoundError:
    df_base_sparsity = pd.DataFrame()
    print("Base MLP sparsity not found")

Base MLP norms: 1230 rows
Base MLP probe: 48 rows
Base neuron selectivity: 82 rows
Base MLP sparsity: 1230 rows


In [14]:
# MLP norm trajectory overlay: base (solid) vs circuit (dashed)
if not df_base_norms.empty and not df_circuit_norms.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        for band in BANDS:
            color = BAND_COLORS.get(band, "gray")
            label_name = BAND_NAMES.get(band, band)

            # Base (solid)
            df_base_mb = (
                df_base_norms[
                    (df_base_norms["model"] == model) & (df_base_norms["band"] == band)
                ]
                .groupby("layer")["mean_norm"]
                .mean()
                .reset_index()
            )
            if not df_base_mb.empty:
                ax.plot(
                    df_base_mb["layer"],
                    df_base_mb["mean_norm"],
                    color=color,
                    linewidth=2,
                    marker="o",
                    ms=3,
                    label=f"{label_name} (base)",
                )

            # Circuit (dashed)
            df_circ_mb = (
                df_circuit_norms[
                    (df_circuit_norms["model"] == model)
                    & (df_circuit_norms["band"] == band)
                ]
                .groupby("layer")["mean_norm"]
                .mean()
                .reset_index()
            )
            if not df_circ_mb.empty:
                ax.plot(
                    df_circ_mb["layer"],
                    df_circ_mb["mean_norm"],
                    color=color,
                    linewidth=2,
                    linestyle="--",
                    marker="s",
                    ms=3,
                    label=f"{label_name} (circuit)",
                )

        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean MLP Output Norm")
        ax.set_title(f"MLP Norm: Base vs Circuit \u2014 {model}")
        ax.legend(fontsize=7, ncol=2)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_05c_06_norm_comparison_{model}.png")
else:
    print("Skipping norm overlay: missing base or circuit data")

In [15]:
# MLP probe trajectory overlay: base vs circuit
if not df_base_probe.empty and not df_circuit_probe.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        # Base (solid blue)
        df_base_m = df_base_probe[df_base_probe["model"] == model]
        if not df_base_m.empty:
            df_mean = (
                df_base_m.groupby("layer")
                .agg({"accuracy": "mean", "std": "mean"})
                .reset_index()
            )
            ax.plot(
                df_mean["layer"],
                df_mean["accuracy"],
                color="steelblue",
                label="Base",
                linewidth=2,
                marker="o",
                ms=4,
            )
            ax.fill_between(
                df_mean["layer"],
                df_mean["accuracy"] - df_mean["std"],
                df_mean["accuracy"] + df_mean["std"],
                color="steelblue",
                alpha=0.15,
            )

        # Circuit (dashed coral)
        df_circ_m = df_circuit_probe[df_circuit_probe["model"] == model]
        if not df_circ_m.empty:
            df_mean = (
                df_circ_m.groupby("layer")
                .agg({"accuracy": "mean", "std": "mean"})
                .reset_index()
            )
            ax.plot(
                df_mean["layer"],
                df_mean["accuracy"],
                color="coral",
                label="Circuit",
                linewidth=2,
                linestyle="--",
                marker="s",
                ms=4,
            )
            ax.fill_between(
                df_mean["layer"],
                df_mean["accuracy"] - df_mean["std"],
                df_mean["accuracy"] + df_mean["std"],
                color="coral",
                alpha=0.15,
            )

        ax.axhline(
            y=1.0 / len(BANDS), color="gray", linestyle=":", alpha=0.5, label="Chance"
        )
        ax.set_xlabel("Layer")
        ax.set_ylabel("MLP Probe Accuracy")
        ax.set_title(f"MLP Probe: Base vs Circuit \u2014 {model}")
        ax.legend()
        ax.set_ylim(0, 1)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_05c_07_probe_comparison_{model}.png")
else:
    print("Skipping probe overlay: missing base or circuit data")

In [16]:
# Neuron selectivity correlation: base vs circuit
# Both use F-statistic or selectivity fraction per layer -- correlate across layers
if not df_base_selectivity.empty and not df_circuit_selectivity.empty:
    rows_sel_corr = []

    for model in MODELS:
        df_base_m = df_base_selectivity[
            df_base_selectivity["model"] == model
        ].sort_values("layer")
        df_circ_m = df_circuit_selectivity[
            df_circuit_selectivity["model"] == model
        ].sort_values("layer")

        if df_base_m.empty or df_circ_m.empty:
            continue

        # Merge on layer
        merged = pd.merge(
            df_base_m[["layer", "frac_selective", "mean_f_stat"]],
            df_circ_m[["layer", "frac_selective_2x", "mean_f_stat_sample"]],
            on="layer",
            how="inner",
            suffixes=("_base", "_circuit"),
        )

        if len(merged) < 3:
            continue

        # Correlate frac_selective (base uses 'frac_selective', circuit uses 'frac_selective_2x')
        rho_frac, p_frac = stats.spearmanr(
            merged["frac_selective"], merged["frac_selective_2x"]
        )
        rho_f, p_f = stats.spearmanr(
            merged["mean_f_stat"], merged["mean_f_stat_sample"]
        )

        rows_sel_corr.append(
            {
                "model": model,
                "spearman_frac_selective": float(rho_frac),
                "p_frac_selective": float(p_frac),
                "spearman_f_stat": float(rho_f),
                "p_f_stat": float(p_f),
                "n_layers_compared": len(merged),
            }
        )

    df_sel_corr = pd.DataFrame(rows_sel_corr)
    save_analysis_comp(df_sel_corr, "05c_selectivity_correlation.csv")
    print("Neuron selectivity correlation (base vs circuit):")
    print(df_sel_corr.to_string(index=False))

    # Scatter plots: base vs circuit selectivity per layer
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for model in MODELS:
        df_base_m = df_base_selectivity[df_base_selectivity["model"] == model]
        df_circ_m = df_circuit_selectivity[df_circuit_selectivity["model"] == model]
        if df_base_m.empty or df_circ_m.empty:
            continue
        merged = pd.merge(
            df_base_m[["layer", "frac_selective", "mean_f_stat"]],
            df_circ_m[["layer", "frac_selective_2x", "mean_f_stat_sample"]],
            on="layer",
            how="inner",
        )
        color = MODEL_COLORS.get(model, "gray")
        axes[0].scatter(
            merged["frac_selective"],
            merged["frac_selective_2x"],
            color=color,
            label=model,
            alpha=0.7,
            s=30,
        )
        axes[1].scatter(
            merged["mean_f_stat"],
            merged["mean_f_stat_sample"],
            color=color,
            label=model,
            alpha=0.7,
            s=30,
        )

    # Add diagonal reference
    for ax in axes:
        lims = [
            min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1]),
        ]
        ax.plot(lims, lims, "k--", alpha=0.3, linewidth=1)
        ax.legend(fontsize=8)

    axes[0].set_xlabel("Base: Frac Selective (p<0.01)")
    axes[0].set_ylabel("Circuit: Frac Selective (>2x)")
    axes[0].set_title("Neuron Selectivity Fraction: Base vs Circuit")

    axes[1].set_xlabel("Base: Mean F-stat")
    axes[1].set_ylabel("Circuit: Mean F-stat (sample)")
    axes[1].set_title("Neuron F-stat: Base vs Circuit")

    fig.tight_layout()
    save_figure_comp(fig, "viz_05c_08_selectivity_correlation.png")
else:
    print("Skipping selectivity correlation: missing base or circuit data")

<TMPDIR>/ipykernel_401534/3036599213.py:25: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_frac, p_frac = stats.spearmanr(merged['frac_selective'], merged['frac_selective_2x'])
<TMPDIR>/ipykernel_401534/3036599213.py:25: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_frac, p_frac = stats.spearmanr(merged['frac_selective'], merged['frac_selective_2x'])
<TMPDIR>/ipykernel_401534/3036599213.py:25: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_frac, p_frac = stats.spearmanr(merged['frac_selective'], merged['frac_selective_2x'])


Neuron selectivity correlation (base vs circuit):
      model  spearman_frac_selective  p_frac_selective  spearman_f_stat     p_f_stat  n_layers_compared
 pythia-70m                      NaN               NaN         0.600000 2.080000e-01                  6
pythia-160m                      NaN               NaN         0.986014 4.116896e-09                 12
pythia-410m                 0.316356          0.132047         0.976522 3.693852e-16                 24
  pythia-1b                      NaN               NaN         0.838235 5.010342e-05                 16
pythia-1.4b                -0.256042          0.227185         0.874783 2.266043e-08                 24


In [17]:
# Sparsity comparison: base vs circuit Gini trajectories
if not df_base_sparsity.empty and not df_circuit_sparsity.empty:
    # Per-model overlay
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        for band in BANDS:
            color = BAND_COLORS.get(band, "gray")
            label_name = BAND_NAMES.get(band, band)

            # Base (solid)
            df_base_mb = (
                df_base_sparsity[
                    (df_base_sparsity["model"] == model)
                    & (df_base_sparsity["band"] == band)
                ]
                .groupby("layer")["mean_gini"]
                .mean()
                .reset_index()
            )
            if not df_base_mb.empty:
                ax.plot(
                    df_base_mb["layer"],
                    df_base_mb["mean_gini"],
                    color=color,
                    linewidth=2,
                    marker="o",
                    ms=3,
                    label=f"{label_name} (base)",
                )

            # Circuit (dashed)
            df_circ_mb = (
                df_circuit_sparsity[
                    (df_circuit_sparsity["model"] == model)
                    & (df_circuit_sparsity["band"] == band)
                ]
                .groupby("layer")["mean_gini"]
                .mean()
                .reset_index()
            )
            if not df_circ_mb.empty:
                ax.plot(
                    df_circ_mb["layer"],
                    df_circ_mb["mean_gini"],
                    color=color,
                    linewidth=2,
                    linestyle="--",
                    marker="s",
                    ms=3,
                    label=f"{label_name} (circuit)",
                )

        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean Gini Coefficient")
        ax.set_title(f"MLP Sparsity: Base vs Circuit \u2014 {model}")
        ax.legend(fontsize=7, ncol=2)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_05c_09_sparsity_comparison_{model}.png")

    # Summary delta heatmap: mean Gini difference (circuit - base) per model x band
    base_agg = (
        df_base_sparsity.groupby(["model", "band"])["mean_gini"]
        .mean()
        .reset_index()
        .rename(columns={"mean_gini": "base_gini"})
    )
    circ_agg = (
        df_circuit_sparsity.groupby(["model", "band"])["mean_gini"]
        .mean()
        .reset_index()
        .rename(columns={"mean_gini": "circuit_gini"})
    )
    merged_gini = pd.merge(base_agg, circ_agg, on=["model", "band"], how="inner")
    merged_gini["delta_gini"] = merged_gini["circuit_gini"] - merged_gini["base_gini"]

    save_analysis_comp(merged_gini, "05c_sparsity_delta.csv")

    pivot_delta = merged_gini.pivot(index="model", columns="band", values="delta_gini")
    pivot_delta = pivot_delta.reindex(index=MODELS, columns=BANDS)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot_delta,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        center=0,
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_title("Sparsity Change (Circuit - Base Gini)")
    ax.set_xlabel("Band")
    ax.set_ylabel("Model")
    fig.tight_layout()
    save_figure_comp(fig, "viz_05c_10_sparsity_delta_heatmap.png")
else:
    print("Skipping sparsity comparison: missing base or circuit data")

## 8. Cross-Model Patterns

Aggregate circuit MLP metrics across model sizes and compare
with base model patterns.

In [18]:
rows_summary = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_mlp = MODEL_D_MLP[model]

    # Circuit probe peak
    df_probe_m = df_circuit_probe[df_circuit_probe["model"] == model]
    if not df_probe_m.empty:
        peak_acc = df_probe_m.groupby("layer")["accuracy"].mean().max()
        peak_layer = int(df_probe_m.groupby("layer")["accuracy"].mean().idxmax())
    else:
        peak_acc, peak_layer = np.nan, np.nan

    # Circuit mean norm (across all bands/draws/layers)
    df_norms_m = df_circuit_norms[df_circuit_norms["model"] == model]
    mean_norm = df_norms_m["mean_norm"].mean() if not df_norms_m.empty else np.nan

    # Circuit sparsity
    df_spar_m = df_circuit_sparsity[df_circuit_sparsity["model"] == model]
    mean_gini = df_spar_m["mean_gini"].mean() if not df_spar_m.empty else np.nan

    # Circuit selectivity
    df_sel_m = df_circuit_selectivity[df_circuit_selectivity["model"] == model]
    mean_frac_sel = (
        df_sel_m["frac_selective_2x"].mean() if not df_sel_m.empty else np.nan
    )

    # Base probe peak for comparison
    base_peak_acc = np.nan
    if not df_base_probe.empty:
        df_base_m = df_base_probe[df_base_probe["model"] == model]
        if not df_base_m.empty:
            base_peak_acc = df_base_m.groupby("layer")["accuracy"].mean().max()

    rows_summary.append(
        {
            "model": model,
            "model_capacity": MODEL_CAPACITY.get(model, 0),
            "d_mlp": d_mlp,
            "n_layers": n_layers,
            "circuit_peak_probe_acc": peak_acc,
            "circuit_peak_probe_layer": peak_layer,
            "base_peak_probe_acc": base_peak_acc,
            "probe_acc_retention": peak_acc / base_peak_acc
            if (
                not np.isnan(peak_acc)
                and not np.isnan(base_peak_acc)
                and base_peak_acc > 0
            )
            else np.nan,
            "circuit_mean_norm": mean_norm,
            "circuit_mean_gini": mean_gini,
            "circuit_mean_frac_selective": mean_frac_sel,
        }
    )

df_summary = pd.DataFrame(rows_summary)
save_analysis_circuit(df_summary, "05c_circuit_mlp_summary.csv")
print("Cross-model circuit MLP summary:")
print(df_summary.to_string(index=False))

Cross-model circuit MLP summary:
      model  model_capacity  d_mlp  n_layers  circuit_peak_probe_acc  circuit_peak_probe_layer  base_peak_probe_acc  probe_acc_retention  circuit_mean_norm  circuit_mean_gini  circuit_mean_frac_selective
 pythia-70m              70   2048         6                0.519111                         5             0.556444             0.932907          15.330964           0.434975                     0.000000
pythia-160m             160   3072        12                0.617481                         7             0.605333             1.020069          18.929911           0.419187                     0.000000
pythia-410m             410   4096        24                0.632889                        23             0.656000             0.964770          13.562851           0.391608                     0.000010
  pythia-1b            1000   8192        16                0.684148                        15             0.637333             1.073454          26.01

In [19]:
# Cross-model scaling: circuit vs base probe accuracy
if not df_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    x = df_summary["model_capacity"]

    # Panel 1: Probe accuracy (base vs circuit)
    ax = axes[0]
    ax.plot(
        x,
        df_summary["base_peak_probe_acc"],
        "o-",
        color="steelblue",
        label="Base",
        linewidth=2,
        ms=6,
    )
    ax.plot(
        x,
        df_summary["circuit_peak_probe_acc"],
        "s--",
        color="coral",
        label="Circuit",
        linewidth=2,
        ms=6,
    )
    ax.axhline(y=1.0 / len(BANDS), color="gray", linestyle=":", alpha=0.5)
    ax.set_title("Peak MLP Probe Accuracy")
    ax.set_ylabel("Accuracy")
    ax.legend()
    ax.set_ylim(0, 1)

    # Panel 2: Circuit sparsity (Gini)
    ax = axes[1]
    ax.plot(
        x,
        df_summary["circuit_mean_gini"],
        "s--",
        color="coral",
        label="Circuit",
        linewidth=2,
        ms=6,
    )
    ax.set_title("Circuit Mean Gini (Sparsity)")
    ax.set_ylabel("Gini Coefficient")
    ax.legend()

    # Panel 3: Probe accuracy retention
    ax = axes[2]
    ax.plot(
        x, df_summary["probe_acc_retention"], "D-", color="#2ca02c", linewidth=2, ms=6
    )
    ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title("Probe Accuracy Retention (Circuit/Base)")
    ax.set_ylabel("Retention Ratio")

    for ax in axes:
        ax.set_xlabel("Model Capacity (M params)")
        ax.set_xscale("log")

    fig.suptitle("Circuit MLP Metrics vs Model Size", y=1.02)
    fig.tight_layout()
    save_figure_circuit(fig, "viz_05c_11_cross_model_scaling.png")

In [20]:
# Norm delta heatmap: (circuit - base) mean MLP norm per model x band
if not df_base_norms.empty and not df_circuit_norms.empty:
    base_norm_agg = (
        df_base_norms.groupby(["model", "band"])["mean_norm"]
        .mean()
        .reset_index()
        .rename(columns={"mean_norm": "base_norm"})
    )
    circ_norm_agg = (
        df_circuit_norms.groupby(["model", "band"])["mean_norm"]
        .mean()
        .reset_index()
        .rename(columns={"mean_norm": "circuit_norm"})
    )
    merged_norm = pd.merge(
        base_norm_agg, circ_norm_agg, on=["model", "band"], how="inner"
    )
    merged_norm["delta_norm"] = merged_norm["circuit_norm"] - merged_norm["base_norm"]
    merged_norm["norm_ratio"] = merged_norm["circuit_norm"] / merged_norm[
        "base_norm"
    ].clip(lower=1e-10)

    save_analysis_comp(merged_norm, "05c_norm_delta.csv")

    pivot_ratio = merged_norm.pivot(index="model", columns="band", values="norm_ratio")
    pivot_ratio = pivot_ratio.reindex(index=MODELS, columns=BANDS)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot_ratio,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=1.0,
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_title("MLP Norm Ratio (Circuit / Base)")
    ax.set_xlabel("Band")
    ax.set_ylabel("Model")
    fig.tight_layout()
    save_figure_comp(fig, "viz_05c_12_norm_ratio_heatmap.png")

In [21]:
print("=" * 70)
print("NOTEBOOK 05c: CIRCUIT MLP ANALYSIS COMPLETE")
print("=" * 70)

print("\n--- Key Findings ---")

# Circuit probe peaks
if not df_circuit_probe.empty:
    print("\nCircuit MLP probe peak accuracy:")
    for model in MODELS:
        df_m = df_circuit_probe[df_circuit_probe["model"] == model]
        if not df_m.empty:
            peak = df_m.groupby("layer")["accuracy"].mean()
            print(f"  {model}: {peak.max():.3f} at layer {peak.idxmax()}")

# Sparsity summary
if not df_circuit_sparsity.empty:
    print("\nCircuit MLP mean Gini (sparsity):")
    for model in MODELS:
        df_m = df_circuit_sparsity[df_circuit_sparsity["model"] == model]
        if not df_m.empty:
            print(f"  {model}: {df_m['mean_gini'].mean():.3f}")

# Selectivity summary
if not df_circuit_selectivity.empty:
    print("\nCircuit neuron selectivity (frac >2x):")
    for model in MODELS:
        df_m = df_circuit_selectivity[df_circuit_selectivity["model"] == model]
        if not df_m.empty:
            print(
                f"  {model}: {df_m['frac_selective_2x'].mean():.3f} (mean across layers)"
            )

# Retention summary
if not df_summary.empty:
    print("\nProbe accuracy retention (circuit/base):")
    for _, row in df_summary.iterrows():
        retention = row.get("probe_acc_retention", np.nan)
        if not np.isnan(retention):
            print(f"  {row['model']}: {retention:.3f}")

print(f"\n--- Output Files ---")
print(f"Circuit analysis: {CIRCUIT_ANALYSIS}")
for f in sorted(CIRCUIT_ANALYSIS.glob("05c_*")):
    print(f"  {f.name}")
print(f"\nComparison analysis: {COMP_ANALYSIS}")
for f in sorted(COMP_ANALYSIS.glob("05c_*")):
    print(f"  {f.name}")
print(f"\nCircuit figures: {CIRCUIT_VIZ}")
for f in sorted(CIRCUIT_VIZ.glob("viz_05c_*")):
    print(f"  {f.name}")
print(f"\nComparison figures: {COMP_VIZ}")
for f in sorted(COMP_VIZ.glob("viz_05c_*")):
    print(f"  {f.name}")

print("\n" + "=" * 70)

NOTEBOOK 05c: CIRCUIT MLP ANALYSIS COMPLETE

--- Key Findings ---

Circuit MLP probe peak accuracy:
  pythia-70m: 0.519 at layer 5
  pythia-160m: 0.617 at layer 7
  pythia-410m: 0.633 at layer 23
  pythia-1b: 0.684 at layer 15
  pythia-1.4b: 0.696 at layer 22

Circuit MLP mean Gini (sparsity):
  pythia-70m: 0.435
  pythia-160m: 0.419
  pythia-410m: 0.392
  pythia-1b: 0.375
  pythia-1.4b: 0.368

Circuit neuron selectivity (frac >2x):
  pythia-70m: 0.000 (mean across layers)
  pythia-160m: 0.000 (mean across layers)
  pythia-410m: 0.000 (mean across layers)
  pythia-1b: 0.000 (mean across layers)
  pythia-1.4b: 0.000 (mean across layers)

Probe accuracy retention (circuit/base):
  pythia-70m: 0.933
  pythia-160m: 1.020
  pythia-410m: 0.965
  pythia-1b: 1.073
  pythia-1.4b: 1.029

--- Output Files ---
Circuit analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/mlp/circuit/analysis
  05c_circuit_mlp_contribution.csv
  05c_circuit_mlp_norms.csv
  05c_circuit_mlp_probe.csv
  05c